**data and preprocessing**

In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
torch.manual_seed(1337)

In [4]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-03-02 16:30:10--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2025-03-02 16:30:10 (32.7 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [5]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [6]:
len(text)

1115394

In [7]:
vocabulary = sorted(set(text))
vocab_size = len(vocabulary)
print(''.join(vocabulary))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [8]:
str_to_int = {ch:i for i,ch in enumerate(vocabulary)}
int_to_str = {i:ch for i,ch in enumerate(vocabulary)}

encode = lambda s: [str_to_int[c] for c in s]
decode = lambda s: ''.join([int_to_str[c] for c in s])

print(encode("hello there"))
print(decode(encode("hello there")))

[46, 43, 50, 50, 53, 1, 58, 46, 43, 56, 43]
hello there


In [9]:
import torch 
import torch.nn
data = torch.tensor(encode(text),dtype=torch.long)
print(data)


tensor([18, 47, 56,  ..., 45,  8,  0])


In [10]:
#split data 
n = int(0.9*len(data))
train = data[:n]
val= data[n:]

In [11]:
block_size = 8
train[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [12]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train if split == 'train' else val
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print([data[i:i+block_size] for i in ix])
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [13]:
# #1. Create a small vocabulary
# vocab = ['<PAD>', 'hello', 'world', 'how', 'are', 'you']
# vocab_size = len(vocab)
# embedding_dim = 3

# # 2. Create word-to-index mapping
# word_to_idx = {word: idx for idx, word in enumerate(vocab)}

# # 3. Create embedding layer
# embedding = nn.Embedding(vocab_size, embedding_dim)

# # Let's look at the embedding weights
# print("Embedding weight matrix shape:", embedding.weight.shape)
# print("Initial embedding matrix:\n", embedding.weight)

# # 4. Convert words to indices
# sentence = ['hello', 'world']
# indices = torch.tensor([word_to_idx[word] for word in sentence])

# # 5. Get embeddings
# vectors = embedding(indices)

# print("\nIndices:", indices)
# print("Resulting embeddings shape:", vectors.shape)
# print("Embeddings:\n", vectors)

# # Verify it's just a lookup
# print("\nVerification:")
# print("Manual lookup of 'hello':", embedding.weight[word_to_idx['hello']])
# print("Embedding lookup of 'hello':", vectors[0])

In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers NB. here B = Batch, T= Context Size, C = class/vocabulary 
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # print(logits.shape)
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            # print(logits.shape)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)

            
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
# print(loss)

print(decode(m.generate(torch.zeros(1,1,dtype= torch.long), max_new_tokens=10)[0].tolist()))

torch.Size([32, 65])

Sr?qP-QWkt


In [15]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [16]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


2.4730143547058105


In [17]:
print(decode(m.generate(torch.zeros(1,1,dtype= torch.long), max_new_tokens=1000)[0].tolist()))


By y a!
Caril.n y, denjxhece w illd CHAL, mer thoun s's:Conchuntilalllevise sthat dy hangilyoteng h hasbe pave pirance
Rie hicomyonthar's
Plinseard ith henoure wounonthioneir thondy, y heltieiengerofo'dsssit ey
KIN d pe wither vouprrouthercc.
hathe; d!
My hind tt hinig t ouchos tes; st yo hind wotte grotonear 'so it t jod weancotha:
h hay.JUCle n prids, r loncave w hollular s O:
HIs; ht anjx?

DUThinqunt.

LaZAnde.
athave l.
KEONH:
ARThanco be y,-hedarwnoddy scace, tridesar, wnl'shenous s ls, theresseys
PlorseelapinghiybHen yof GLUCEN t l-t E:
I hisgothers je are!-e!
QLYotouciullle'z,
Thitertho s?
NDan'spererfo cist ripl chys er orlese;
Yo jehof h hecere ek? wferommot mowo soaf yoit, ince his, t, f at. fal whetrimy bupof tor atha Bu!
JOutho f cimimave.
NEDUSt cir selle p wie wede
Ro n apenor f'Y tover witys an sh d w t e w!
CEOntiretoaveE IINpe, theck. cung.
ORIsthies hacin benqurd bll, d a r w wistatsowor ath
Fivet bloll ang a-I theeancu,
LINCI'T:
Sarry t I Ane sze t
LCKI thit,
n.
Fa

-----Bigram End-----

In [18]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x

tensor([[[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]],

        [[ 1.3488, -0.1396],
         [ 0.2858,  0.9651],
         [-2.0371,  0.4931],
         [ 1.4870,  0.5910],
         [ 0.1260, -1.5627],
         [-1.1601, -0.3348],
         [ 0.4478, -0.8016],
         [ 1.5236,  2.5086]],

        [[-0.6631, -0.2513],
         [ 1.0101,  0.1215],
         [ 0.1584,  1.1340],
         [-1.1539, -0.2984],
         [-0.5075, -0.9239],
         [ 0.5467, -1.4948],
         [-1.2057,  0.5718],
         [-0.5974, -0.6937]],

        [[ 1.6455, -0.8030],
         [ 1.3514, -0.2759],
         [-1.5108,  2.1048],
         [ 2.7630, -1.7465],
         [ 1.4516, -1.5103],
         [ 0.8212, -0.2115],
         [ 0.7789,  1.5333],
         [ 1.6097, -0.4032]]])

In [19]:
weight = torch.tril(torch.ones(T,T))
weight = weight/weight.sum(1,keepdim=True)
weight

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [66]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 2,4,4 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 6
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
print('k',k.transpose(-2,-1))
q = query(x) # (B, T, 16)
print('q',q)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

print(wei)
tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei)

v = value(x)
out = wei @ v
#out = wei @ x

out

k tensor([[[-0.1821, -0.1370, -0.2952,  0.2693],
         [ 0.2057,  0.0662,  0.1047, -0.4609],
         [-0.0849, -0.3146,  0.7144,  0.7683],
         [-0.1182,  0.2431, -0.5965,  0.3820],
         [ 0.3824,  0.4389,  0.3984, -0.8455],
         [-0.4365,  0.6680, -0.1796, -0.0424]],

        [[-0.0947,  0.4319,  0.1419,  0.4275],
         [-0.2001, -0.0643, -0.0778, -0.5580],
         [ 0.2446, -0.5801,  0.0889, -0.1019],
         [ 0.6253, -0.5390,  0.5491,  1.0433],
         [ 0.1506, -0.6460, -0.4678, -0.8134],
         [ 0.7213,  0.6033, -1.3687,  1.2482]]], grad_fn=<TransposeBackward0>)
q tensor([[[-0.0284,  0.1309,  0.5184,  0.2507,  0.2316,  0.1183],
         [-0.6241, -0.4524, -0.2224, -0.3293, -0.3268,  0.0348],
         [ 0.5666,  0.0489,  0.6693,  0.5976,  0.7891, -0.1821],
         [ 0.4090,  0.5169,  0.1501,  0.4076, -0.3023,  0.1478]],

        [[-0.5609, -0.1304,  0.1708,  0.0766, -0.5945,  0.2551],
         [ 0.3164, -0.6401, -1.8101, -1.1252,  0.1136, -0.7021],
      

tensor([[[ 0.2188,  0.3971, -0.2551, -0.2845, -0.0386, -0.3486],
         [ 0.2316,  0.2743, -0.4691,  0.0400, -0.1414, -0.4435],
         [ 0.2748,  0.3136, -0.1839, -0.1623,  0.1719, -0.0630],
         [ 0.1349, -0.0411,  0.1278, -0.0678,  0.3649,  0.1095]],

        [[ 0.4655, -0.0675, -0.4361,  0.3194,  0.4508, -0.4643],
         [-0.9604, -0.6205,  0.5094,  0.5500, -0.8969,  0.9507],
         [ 0.1326, -0.1555, -0.1348,  0.1969,  0.1784, -0.2349],
         [-0.6643, -0.6037,  0.3470,  0.5069, -0.5497,  0.5263]]],
       grad_fn=<UnsafeViewBackward0>)

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(555)

In [15]:

token_embedding_table = nn.Embedding(5, 2)
positional_embedding_table = nn.Embedding(2,2)
idx = torch.randint(1,(1,5,2))
token_embed = token_embedding_table(idx) # (B,T,C)
pos_embed = positional_embedding_table(torch.arange(2)) # T,C
x = token_embed + pos_embed
print(token_embed.shape,pos_embed.shape,x.shape)
print(token_embed,pos_embed,x)


torch.Size([1, 5, 2, 2]) torch.Size([2, 2]) torch.Size([1, 5, 2, 2])
tensor([[[[ 0.3451, -0.8717],
          [ 0.3451, -0.8717]],

         [[ 0.3451, -0.8717],
          [ 0.3451, -0.8717]],

         [[ 0.3451, -0.8717],
          [ 0.3451, -0.8717]],

         [[ 0.3451, -0.8717],
          [ 0.3451, -0.8717]],

         [[ 0.3451, -0.8717],
          [ 0.3451, -0.8717]]]], grad_fn=<EmbeddingBackward0>) tensor([[-0.7371, -0.4538],
        [-0.6063,  0.8926]], grad_fn=<EmbeddingBackward0>) tensor([[[[-0.3920, -1.3255],
          [-0.2612,  0.0209]],

         [[-0.3920, -1.3255],
          [-0.2612,  0.0209]],

         [[-0.3920, -1.3255],
          [-0.2612,  0.0209]],

         [[-0.3920, -1.3255],
          [-0.2612,  0.0209]],

         [[-0.3920, -1.3255],
          [-0.2612,  0.0209]]]], grad_fn=<AddBackward0>)


In [57]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3 # self attention can't take high learning rate.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 32
# ------------

torch.manual_seed(555)

# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embed,head_size,bias=False)
        self.key = nn.Linear(n_embed,head_size,bias=False)
        self.value = nn.Linear(n_embed,head_size,bias=False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
    
    def forward(self,x):
        B,T,C = x.shape
        q = self.query(x)
        k = self.key(x)
        weight = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T) Scaled dot product attention 
        weight= weight.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T) lower triangular matrix
        weight= F.softmax(weight, dim=-1) # (B, T, T)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = weight @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out
        

class SingleHeadLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.positional_embedding_table = nn.Embedding(block_size,n_embed)
        self.self_atte_head = Head(n_embed)
        self.linear_head = nn.Linear(n_embed, vocab_size)
        

    def forward(self, idx, targets=None):

        B,T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        token_embed = self.token_embedding_table(idx) # (B,T,C)
        pos_embed = self.positional_embedding_table(torch.arange(T,device=device)) # T,C
        x = token_embed + pos_embed
        x = self.self_atte_head(x)
        logits = self.linear_head(x)

        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last_block_size tokens
            idx_cond = idx[:,-block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = SingleHeadLanguageModel()
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
# context = torch.zeros((1, 1), dtype=torch.long, device=device)
# print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 4.1836, val loss 4.1842
step 100: train loss 3.2177, val loss 3.2457
step 200: train loss 3.0570, val loss 3.0828
step 300: train loss 2.9541, val loss 2.9739
step 400: train loss 2.8826, val loss 2.8985
step 500: train loss 2.8410, val loss 2.8402
step 600: train loss 2.8147, val loss 2.8269
step 700: train loss 2.7856, val loss 2.7897
step 800: train loss 2.7632, val loss 2.7749
step 900: train loss 2.7485, val loss 2.7561
step 1000: train loss 2.7357, val loss 2.7445
step 1100: train loss 2.7228, val loss 2.7189
step 1200: train loss 2.7182, val loss 2.7133
step 1300: train loss 2.6950, val loss 2.6961
step 1400: train loss 2.6906, val loss 2.6990
step 1500: train loss 2.6765, val loss 2.6794
step 1600: train loss 2.6683, val loss 2.6638
step 1700: train loss 2.6632, val loss 2.6622
step 1800: train loss 2.6521, val loss 2.6574
step 1900: train loss 2.6370, val loss 2.6365
step 2000: train loss 2.6291, val loss 2.6378
step 2100: train loss 2.6166, val loss 2.6209


In [58]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


H
an isswat aw KNUSAsthart; ren't ut sinivan;
Fsry avelinind, olit,o ht ay, ota fsed be sht it beinsiesl this,o prdp
FAEN shinorct thhos hy owotir and E:
PI: Irrk cofk bien ted sld Vorknd wathe cant G:
At sphl, ffe:
Tho ow liee po ng'or war inlg cud lel co.
Thulofu
FAp'd: hwepl che winay won.

ANGTou veewake allise, fslen ther hik hise, ssou y, haju thee homours, ake st osininfe' den.
NGUKYDrices, my wingory on: yos wacely mee wenoong anter sipet GRould busisld' fown ay mid, briat tharebe.

Tuld
